## Import Needed Package

In [19]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

## Load Dataset

In [20]:
    from google.colab import drive
    drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
data = pd.read_csv("/content/drive/MyDrive/notes/IMDB Dataset.csv")

In [22]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [23]:
data.shape

(50000, 2)

In [24]:
data.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [25]:
data["sentiment"].value_counts

<bound method IndexOpsMixin.value_counts of 0        positive
1        positive
2        positive
3        negative
4        positive
           ...   
49995    positive
49996    negative
49997    negative
49998    negative
49999    negative
Name: sentiment, Length: 50000, dtype: object>

## One Hot Encoding

### Label Encoder

In [26]:
data.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

In [27]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


## Data Preprocessing

In [28]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [29]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [30]:
train_data.shape

(40000, 2)

In [31]:
test_data.shape

(10000, 2)

In [32]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data['review'])

In [34]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen = 200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data['review']), maxlen = 200)

In [35]:
X_train

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [36]:
X_test

array([[   0,    0,    0, ...,  995,  719,  155],
       [  12,  162,   59, ...,  380,    7,    7],
       [   0,    0,    0, ...,   50, 1088,   96],
       ...,
       [   0,    0,    0, ...,  125,  200, 3241],
       [   0,    0,    0, ..., 1066,    1, 2305],
       [   0,    0,    0, ...,    1,  332,   27]], dtype=int32)

In [37]:
Y_train = train_data['sentiment']
Y_test = test_data['sentiment']

In [38]:
Y_train

,sentiment
39087,0
30893,0
45278,1
16398,0
13653,0
...,...
11284,1
44732,1
38158,0
860,1


In [39]:
Y_test

,sentiment
33553,1
9427,1
199,0
12447,1
39489,0
...,...
28567,0
25079,1
18707,1
15200,0


## Model Building

In [40]:
model = Sequential()
model.add(Embedding(input_dim = 5000, output_dim = 128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

In [41]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [42]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [45]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split = 0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 209s 417ms/step - accuracy: 0.9224 - loss: 0.2036 - val_accuracy: 0.8677 - val_loss: 0.3199
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 220s 439ms/step - accuracy: 0.9265 - loss: 0.1970 - val_accuracy: 0.8700 - val_loss: 0.3457
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 219s 437ms/step - accuracy: 0.9323 - loss: 0.1797 - val_accuracy: 0.8764 - val_loss: 0.3483
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 214s 427ms/step - accuracy: 0.9425 - loss: 0.1580 - val_accuracy: 0.8769 - val_loss: 0.3641
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 208s 416ms/step - accuracy: 0.9473 - loss: 0.1384 - val_accuracy: 0.8740 - val_loss: 0.3533


In [46]:
loss, accuracy = model.evaluate(X_test, Y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 121ms/step - accuracy: 0.8719 - loss: 0.3373


In [48]:
print(loss)

0.3364813029766083


In [49]:
print(accuracy)

0.8754000067710876


## Model Evaluation

### Building a Predictive System

In [50]:
def predictive_system(review):
  # tokenize and pad the review
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen=200)
  prediction = model.predict(padded_sequence)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  return sentiment

In [51]:
predictive_system("This movie was fantastic and amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 443ms/step


'positive'

In [52]:
predictive_system("A trilling adventure with stunning visual")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step


'positive'

In [53]:
predictive_system("A visual masterpiece")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step


'positive'

In [54]:
predictive_system("Overall long and slow")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step


'negative'

In [55]:
# example usage
new_review = "This movie was fantastic. I loved it."
sentiment = predictive_system(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
The sentiment of the review is: positive


In [56]:
# example usage
new_review = "This movie was not that good"
sentiment = predictive_system(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step
The sentiment of the review is: negative


In [57]:
# example usage
new_review = "This movie was ok but not that good."
sentiment = predictive_system(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
The sentiment of the review is: negative


In [58]:
# example usage
new_review = "save your time and money"
sentiment = predictive_system(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
The sentiment of the review is: negative


## Saving Model

In [59]:
model.save('Sentiment_Analysis_IMDB_Reviews.h5')

In [60]:
import joblib
joblib.dump(tokenizer, 'tokenizer.pkl')

['tokenizer.pkl']